# DSS Stage 6.3 — Full XGBoost tuning on Colab
Runs the locked 20-trial search for static and time-aware XGBoost across six validation cells. No test evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REPO_URL = 'REPLACE_WITH_REPOSITORY_URL'
EXPECTED_COMMIT = 'REPLACE_WITH_STAGE6_3A_COMMIT'
DATASET_ROOT = '/content/drive/MyDrive/DSS/medajob-temporal-v0.3'
OUTPUT_DIR = '/content/drive/MyDrive/DSS/stage6_3_colab'
EXPECTED_MANIFEST_SHA256 = 'a6d983e754f49a9e2076022d48109516a931def7c17b1a725394245872aad287'
assert not EXPECTED_COMMIT.startswith('REPLACE_')
assert not REPO_URL.startswith('REPLACE_')


In [ ]:
import os, pathlib, subprocess
repo = pathlib.Path('/content/Stage6')
if not repo.exists(): subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
subprocess.run(['git', 'fetch', '--all', '--tags'], cwd=repo, check=True)
subprocess.run(['git', 'checkout', '--detach', EXPECTED_COMMIT], cwd=repo, check=True)
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()
assert actual == EXPECTED_COMMIT, (actual, EXPECTED_COMMIT)
os.chdir(repo)


In [ ]:
import hashlib, pathlib
manifest = pathlib.Path(DATASET_ROOT) / 'data/candidate55/v0_3/dataset_manifest.json'
actual_manifest_sha = hashlib.sha256(manifest.read_bytes()).hexdigest()
assert actual_manifest_sha == EXPECTED_MANIFEST_SHA256, actual_manifest_sha
print('commit', EXPECTED_COMMIT, 'manifest', actual_manifest_sha)


In [ ]:
import pathlib
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
!python -m pip install -r requirements-colab.txt
!python -m pip install --no-deps -e .
import subprocess, sys
tests = subprocess.run([sys.executable, '-m', 'pytest', '-q'], capture_output=True, text=True)
(pathlib.Path(OUTPUT_DIR) / 'pre_training_tests.log').write_text(f'exit_code={tests.returncode}\n{tests.stdout}{tests.stderr}')
print(tests.stdout)
tests.check_returncode()


In [ ]:
import pathlib, subprocess, sys
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
command = [sys.executable, 'scripts/run_stage6_3.py', '--mode', 'colab-full', '--dataset-root', DATASET_ROOT, '--output-dir', OUTPUT_DIR, '--resume']
subprocess.run(command, check=True)
